Good. Now we connect **geometry (sphere)** + **material (what happens after hit)**.

Before:

```text
Ray hits sphere → only position + normal
```

Now:

```text
Ray hits sphere → position + normal + material
```

That is the purpose of `HitRecord`.

---

# 1. Why `HitRecord`?

Without it:

```python
hit(p, normal, t, front_face, material)
```

too many variables.

Instead:

```python
hit(record)
```

Cleaner.

Mathematically, after solving ray-object intersection:

We store:

Hit point:

$$
P = O + tD
$$

Normal:

$$
N = \frac{P-C}{r}
$$

Material:

$$
M
$$

Front face:

$$
front = D \cdot N < 0
$$

---

# Updated Python Code (with comments + what changed)

---

## hit_record.py

```python
from util.vec3 import dot


class HitRecord:
    def __init__(self):
        # Hit point (NEW: stores where ray hit)
        self.p = None

        # Surface normal (already existed)
        self.normal = None

        # NEW: material pointer/reference
        # Before: no material info
        # Now: allows ray_color() to call material.scatter()
        self.mat = None

        # Distance along ray
        self.t = 0.0

        # NEW: tells whether ray hit outside or inside
        self.front_face = False

    def copy(self, other):
        # Copy all values from another hit record

        self.p = other.p
        self.normal = other.normal

        # NEW: copy material too
        self.mat = other.mat

        self.t = other.t
        self.front_face = other.front_face

    def set_face_normal(self, ray, outward_normal):
        """
        NEW FUNCTION IMPROVEMENT:
        Before:
            rec.normal = outward_normal

        Problem:
            if ray comes from inside sphere,
            normal direction becomes wrong.

        Now:
            normal always opposes incoming ray.
        """

        # Formula:
        # front_face = D.N < 0
        self.front_face = dot(ray.direction(), outward_normal) < 0

        # If outside → normal stays same
        if self.front_face:
            self.normal = outward_normal
        else:
            # If inside → flip normal
            self.normal = outward_normal * -1
```

---

# What changed in `HitRecord`

| Before             | After          | Why                  |
| ------------------ | -------------- | -------------------- |
| `p`                | `p`            | same                 |
| `normal`           | `normal`       | same                 |
| ❌ no `mat`         | ✅ `mat`        | stores material      |
| `t`                | `t`            | same                 |
| `front_face` basic | improved logic | fixes inside/outside |

---

---

## sphere.py

```python
import math
from util.hittable import Hittable
from util.vec3 import dot


class Sphere(Hittable):

    def __init__(self, center, radius, material):
        """
        CHANGED:
        Before:
            Sphere(center, radius)

        Now:
            Sphere(center, radius, material)

        Why:
            Sphere must know what material it has.
        """
        self.center = center
        self.radius = max(0.0, radius)

        # NEW: store material
        self.mat = material

    def hit(self, r, ray_t, rec):

        """
        Sphere equation:

        $$ (P-C)\cdot(P-C)=r^2 $$

        Ray equation:

        $$ P=O+tD $$

        Substitute:

        $$ (O+tD-C)\cdot(O+tD-C)=r^2 $$
        """

        # Vector from ray origin to center
        oc = self.center - r.origin()

        # a = D.D
        a = dot(r.direction(), r.direction())

        # h = D.oc
        h = dot(r.direction(), oc)

        # c = oc.oc - r²
        c = dot(oc, oc) - self.radius * self.radius

        # discriminant
        # CHANGED:
        # optimized form
        # Before: b²-4ac
        # Now: h²-ac
        discriminant = h * h - a * c

        if discriminant < 0:
            # no real root → no hit
            return False

        sqrtd = math.sqrt(discriminant)

        # nearest root
        root = (h - sqrtd) / a

        if not ray_t.surrounds(root):
            root = (h + sqrtd) / a

            if not ray_t.surrounds(root):
                return False

        # store hit distance
        rec.t = root

        # hit point:
        # P = O + tD
        rec.p = r.at(root)

        # outward normal:
        # N = (P-C)/r
        outward_normal = (rec.p - self.center) / self.radius

        """
        CHANGED:
        Before:
            rec.normal = outward_normal

        Problem:
            wrong when ray starts inside sphere

        Now:
            set_face_normal() fixes direction
        """
        rec.set_face_normal(r, outward_normal)

        """
        NEW:
        assign material to hit record
        This connects geometry → material behavior
        """
        rec.mat = self.mat

        return True
```

---

# Mathematical flow

---

## Step 1: Ray

$$
R(t)=O+tD
$$

| Symbol | Meaning   |
| ------ | --------- |
| $O$    | origin    |
| $D$    | direction |
| $t$    | distance  |

---

## Step 2: Sphere

$$
(P-C)^2=r^2
$$

| Symbol | Meaning         |
| ------ | --------------- |
| $P$    | point on sphere |
| $C$    | center          |
| $r$    | radius          |

---

## Step 3: Substitute ray into sphere

$$
(O+tD-C)^2=r^2
$$

Expand:

$$
at^2-2ht+c=0
$$

where:

$$
a=D\cdot D
$$

$$
h=D\cdot(C-O)
$$

$$
c=(C-O)\cdot(C-O)-r^2
$$

---

## Step 4: Solve roots

$$
t=\frac{h\pm \sqrt{h^2-ac}}{a}
$$

---

## Step 5: Hit point

$$
P=O+tD
$$

---

## Step 6: Surface normal

$$
N=\frac{P-C}{r}
$$

---

## Step 7: Face direction

$$
front_face=(D\cdot N)<0
$$

If false:

$$
N=-N
$$

---

# Full data flow

```text
Ray
 ↓
Sphere.hit()
 ↓
Solve quadratic
 ↓
Found t
 ↓
Compute p
 ↓
Compute normal
 ↓
Fix normal direction
 ↓
Attach material
 ↓
Return HitRecord
 ↓
ray_color()
 ↓
rec.mat.scatter()
```

Core idea:

$$
Geometry \rightarrow HitRecord \rightarrow Material \rightarrow ScatteredRay
$$

This is the bridge between **shape** and **light physics**.
